# Model Decision Geometry Comparison

Every classification algorithm has a characteristic inductive bias that shapes
the decision boundary it can represent — linear separators, kernel hyperplanes,
piecewise-constant tree boundaries, smooth neural manifolds.

This notebook trains six model families on the same dataset and visualises
their decision geometries using sensitivity projection.  Because the projection
axes are derived from each model's own Jacobians, the visualisations show how
each model partitions its most decision-sensitive subspace — a direct window
into its inductive bias.

In [ ]:
!pip install -q geolatent

In [ ]:
import numpy as np
import plotly.io as pio
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from geolatent import visualize_decision_geometry

pio.renderers.default = "colab"

---
## Dataset

400 samples, 20 features (10 informative, 5 redundant, 5 noise), 4 classes.
All models are trained on the identical split.  Training accuracy is printed
so the geometry differences can be interpreted in context.

In [ ]:
X, y = make_classification(
    n_samples=400, n_features=20, n_classes=4,
    n_informative=10, n_redundant=5, n_clusters_per_class=1,
    random_state=42,
)
feature_names = [f"f{i:02d}" for i in range(20)]
class_names = {0: "Alpha", 1: "Beta", 2: "Gamma", 3: "Delta"}

def build(estimator):
    return Pipeline([("s", StandardScaler()), ("m", estimator)]).fit(X, y)

models = {
    "Logistic Regression":    build(LogisticRegression(max_iter=500, random_state=0)),
    "Linear SVM":             build(SVC(kernel="linear", probability=True, random_state=0)),
    "RBF SVM":                build(SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=0)),
    "Random Forest":          build(RandomForestClassifier(n_estimators=200, random_state=0)),
    "Gradient Boosting":      build(GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=0)),
    "MLP (256, 128, 64)": build(MLPClassifier(hidden_layer_sizes=(256, 128, 64), max_iter=400, random_state=0)),
}

for name, clf in models.items():
    print(f"{name:<28}  train acc = {clf.score(X, y):.3f}")

---
## Logistic Regression

Linear decision boundaries — each class is separated by a hyperplane.
The three sensitivity axes will span the directions where the log-odds
change fastest, effectively showing the top principal components of the
weight matrix.

In [ ]:
visualize_decision_geometry(
    models["Logistic Regression"], X, y,
    projection_method="sensitivity",
    feature_names=feature_names,
    class_names=class_names,
    show_confidence=True, show_centroids=True, show_ellipsoids=True,
    title="Logistic Regression — linear boundaries",
).show()

---
## Linear SVM vs RBF SVM

The linear SVM finds the maximum-margin hyperplane; the RBF kernel lifts
the data to a higher-dimensional space where a linear separator becomes
a curved boundary in input space.  The difference in boundary smoothness
is visible directly in the isosurface geometry.

In [ ]:
for kernel in ("Linear SVM", "RBF SVM"):
    visualize_decision_geometry(
        models[kernel], X, y,
        projection_method="sensitivity",
        feature_names=feature_names,
        class_names=class_names,
        show_confidence=True, show_centroids=True, show_ellipsoids=True,
        title=f"{kernel} — decision geometry",
    ).show()

---
## Random Forest vs Gradient Boosting

Both are ensemble tree methods, but their boundary characters differ.
Random Forest averages many deep trees over bootstrap samples — smooth
probability estimates near decision boundaries.  Gradient Boosting fits
shallow residual trees sequentially — sharper transitions, more confident
probability surfaces.

In [ ]:
for name in ("Random Forest", "Gradient Boosting"):
    visualize_decision_geometry(
        models[name], X, y,
        projection_method="sensitivity",
        feature_names=feature_names,
        class_names=class_names,
        show_confidence=True, show_centroids=True, show_ellipsoids=True,
        title=f"{name} — decision geometry",
    ).show()

---
## MLP (256, 128, 64)

Deep neural boundary — hierarchical feature composition through three hidden
layers.  Compared to trees, the MLP boundary is differentiable everywhere;
the probability surfaces are smooth and the confidence regions are wider
because the network is more calibrated in interpolation regions.

In [ ]:
visualize_decision_geometry(
    models["MLP (256, 128, 64)"], X, y,
    projection_method="sensitivity",
    feature_names=feature_names,
    class_names=class_names,
    show_confidence=True, show_centroids=True, show_ellipsoids=True,
    title="MLP (256, 128, 64) — deep neural boundary",
).show()

---
## All Six on a Shared Dataset — Summary

Below all models render with PCA projection instead of sensitivity,
so the spatial coordinates are identical across figures and the boundary
shapes can be compared directly in the same geometric reference frame.

In [ ]:
for name, clf in models.items():
    visualize_decision_geometry(
        clf, X, y,
        projection_method="pca",
        feature_names=feature_names,
        class_names=class_names,
        show_confidence=True, show_centroids=True, show_ellipsoids=False,
        title=f"{name} — PCA frame (shared reference)",
    ).show()